<a href="https://colab.research.google.com/github/talhanoor23/generative-ai/blob/main/RAGS/decomposition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [ ]:
docs = "France is developed country. Its capital is Paris"

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
splits = splitter.split_documents([Document(page_content=docs)])

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma
embeddings = OpenAIEmbeddings()
db = Chroma.from_documents(splits, embeddings)
retreiver = db.as_retriever()

In [ ]:
from langchain.prompts import ChatPromptTemplate

# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [ ]:
from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
from langchain_core.output_parsers import StrOutputParser
generate_queries = (prompt_decomposition | llm | StrOutputParser() | (lambda x : x.split("\n")))

In [ ]:
question = "What is the capital of France?"
questions = generate_queries.invoke({"question": question})

In [ ]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question:

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [ ]:
from operator import itemgettter
q_a_pairs = ""
for q in questions:
    rag_chain = (
        {
            "context" = itemgetter("question") | retreiver
            "question" = itemgetter("question")
            "q_a_pairs" = itemgetter("q_a_pairs")
        }
        | decomposition_prompt
        | llm
        | StrOutputParser()
    )
    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = f"Question: {q}\nAnswer: {answer}\n\n"
    q_a_pairs = q_a_pairs + "\n---\n"+ q_a_pair


